# XGBoost Model Evaluation and Prior Case-History Ablation

This clean reproducibility notebook documents the predictive modelling and controlled ablation reported in the dissertation. It excludes restricted surveillance data, credentials, exploratory cells and local machine-specific paths.

**Analytical environment:** Anaconda Python 3.13.9; pandas 2.3.3; NumPy 2.3.5; scikit-learn 1.7.2; XGBoost 3.2.0; SHAP 0.51.0.

Set the `GEOAI_DATA_PATH` environment variable to an authorised CSV matching the schema in `DATA_SCHEMA.md`.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from xgboost import XGBClassifier

DATA_PATH = Path(os.environ.get(
    "GEOAI_DATA_PATH",
    "data/restricted/master_geoai_full_panel_dataset_spatial_temp_clean.csv",
))

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Provide an authorised dataset by setting GEOAI_DATA_PATH. "
        "Restricted source data are not distributed with this package."
    )

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
print("Loaded observations:", len(df))

## Data preparation and leakage-controlled temporal features

The target uses the study-specific threshold of five reported cases. Lagged predictors are calculated within district after chronological sorting. Current-period values are excluded from all case-history predictors.

In [ ]:
df["Date"] = pd.to_datetime(
    {"year": df["Year"], "month": df["Month"], "day": 1},
    errors="coerce",
)
df = df.sort_values(["adm2_pcode", "Date"]).reset_index(drop=True)

FINAL_THRESHOLD = 5
df["Outbreak_Risk"] = (df["COUNT_OBJECTID"] >= FINAL_THRESHOLD).astype(int)
df["Population_Density"] = df["population"] / df["area_sqkm"]

df["Cases_Lag1"] = df.groupby("adm2_pcode")["COUNT_OBJECTID"].shift(1)
df["Incidence_Lag1"] = df.groupby("adm2_pcode")["Incidence_100k"].shift(1)
df["Rainfall_Lag1"] = df.groupby("adm2_pcode")["Rainfall_mm"].shift(1)
df["Temperature_Lag1"] = df.groupby("adm2_pcode")["Temperature_C"].shift(1)
df["Cases_Prior3mo_Avg"] = (
    df.groupby("adm2_pcode")["COUNT_OBJECTID"]
      .transform(lambda s: s.shift(1).rolling(window=3, min_periods=1).mean())
)
df["Wet_Season"] = df["Month"].isin([5, 6, 7, 8, 9, 10]).astype(int)

lag_columns = [
    "Cases_Lag1", "Incidence_Lag1", "Rainfall_Lag1",
    "Temperature_Lag1", "Cases_Prior3mo_Avg",
]
model_df = df.dropna(subset=lag_columns).copy()

features = [
    "Rainfall_mm", "Temperature_C", "Population_Density",
    "Cases_Lag1", "Incidence_Lag1", "Rainfall_Lag1",
    "Temperature_Lag1", "Cases_Prior3mo_Avg", "Wet_Season",
]
X = model_df[features].copy()
y = model_df["Outbreak_Risk"].copy()

assert X.shape == (4760, 9)
assert y.value_counts().to_dict() == {0: 4610, 1: 150}
print("Modelling matrix:", X.shape)
print("Class distribution:", y.value_counts().sort_index().to_dict())

## Controlled train-test split and baseline model

Both the full and ablated models use the same stratified 70:30 split (`random_state=42`) and the same XGBoost configuration. The baseline confusion-matrix assertion prevents accidental use of a cross-validation, repeated-seed or deployment-fitted model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

assert X_train.shape == (3332, 9)
assert X_test.shape == (1428, 9)
assert y_test.value_counts().to_dict() == {0: 1383, 1: 45}

full_model = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
)
full_model.fit(X_train, y_train)

baseline_confusion = confusion_matrix(
    y_test,
    full_model.predict(X_test),
    labels=[0, 1],
)
assert baseline_confusion.tolist() == [[1376, 7], [26, 19]]
print("Baseline confusion matrix:")
print(baseline_confusion)

## Prior case-history ablation

The ablation removes `Cases_Lag1`, `Incidence_Lag1` and `Cases_Prior3mo_Avg`. It retains rainfall, temperature, population density, lagged environmental variables and seasonality. Observation indices and model parameters are held constant.

In [ ]:
case_history_features = [
    "Cases_Lag1",
    "Incidence_Lag1",
    "Cases_Prior3mo_Avg",
]

assert set(case_history_features).issubset(X_train.columns)

X_train_ablated = X_train.drop(columns=case_history_features).copy()
X_test_ablated = X_test.drop(columns=case_history_features).copy()

assert X_train_ablated.index.equals(X_train.index)
assert X_test_ablated.index.equals(X_test.index)
assert X_train_ablated.shape == (3332, 6)
assert X_test_ablated.shape == (1428, 6)
assert set(case_history_features).isdisjoint(X_train_ablated.columns)

ablated_model = clone(full_model)
assert ablated_model.get_params() == full_model.get_params()
ablated_model.fit(X_train_ablated, y_train)

print("Removed predictors:", case_history_features)
print("Retained predictors:", list(X_train_ablated.columns))

In [ ]:
def evaluate_model(fitted_model, X_evaluation, y_evaluation):
    probability = fitted_model.predict_proba(X_evaluation)[:, 1]
    prediction = fitted_model.predict(X_evaluation)
    tn, fp, fn, tp = confusion_matrix(
        y_evaluation, prediction, labels=[0, 1]
    ).ravel()
    return {
        "ROC-AUC": roc_auc_score(y_evaluation, probability),
        "PR-AUC": average_precision_score(y_evaluation, probability),
        "Precision": precision_score(y_evaluation, prediction, zero_division=0),
        "Recall": recall_score(y_evaluation, prediction, zero_division=0),
        "F1": f1_score(y_evaluation, prediction, zero_division=0),
        "TP": int(tp), "FP": int(fp), "FN": int(fn), "TN": int(tn),
    }

comparison = pd.DataFrame(
    [
        evaluate_model(full_model, X_test, y_test),
        evaluate_model(ablated_model, X_test_ablated, y_test),
    ],
    index=[
        "Full predictor set",
        "Without prior case-history predictors",
    ],
)

print(comparison.round(3))

expected_rounded = {
    "Full predictor set": {
        "ROC-AUC": 0.920, "PR-AUC": 0.552, "Precision": 0.731,
        "Recall": 0.422, "F1": 0.535, "TP": 19, "FP": 7,
        "FN": 26, "TN": 1376,
    },
    "Without prior case-history predictors": {
        "ROC-AUC": 0.903, "PR-AUC": 0.492, "Precision": 0.765,
        "Recall": 0.289, "F1": 0.419, "TP": 13, "FP": 4,
        "FN": 32, "TN": 1379,
    },
}

for row, expected in expected_rounded.items():
    for metric, value in expected.items():
        actual = comparison.loc[row, metric]
        assert round(float(actual), 3) == value, (row, metric, actual, value)

comparison.to_csv("xgboost_case_history_ablation_results.csv")
print("Verified dissertation ablation results and saved comparison CSV.")

## Interpretation boundary

The comparison evaluates dependence on prior case-history predictors under the same internal random stratified split. It does not establish causal effects, temporal generalisability, geographic transferability or long-range forecasting performance. Restricted data are required to rerun the notebook and are not distributed in this package.